# `TeacherForcing` model wrapper

The `shap.models.TeacherForcing` class is a specialized tool for calculating scores (log-odds) for text-based models. This is super helpful when you want to explain things like translation or summarization models.

It uses a technique called "teacher forcing," where we feed the exact target sentence we want to explain into the model to see how likely it was to produce those specific words.

In [1]:
import numpy as np
import transformers

import shap

# Let's grab a small encoder-decoder model to play with.
model_name = "patrickvonplaten/t5-tiny-random"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSeq2SeqLM.from_pretrained(model_name)

## Getting started

To use it, just pass your model and tokenizer to the wrapper. SHAP will then take care of managing the target sequences for you.

In [2]:
tf_model = shap.models.TeacherForcing(model, tokenizer)

source_sentence = "Translate this to French: Hello"
target_sentence = "Bonjour"

# We'll wrap these in arrays to stay consistent with SHAP's input style.
X = np.array([source_sentence])
Y = np.array([target_sentence])

# The wrapper gives us the log-odds for every token in our target sentence.
logodds = tf_model(X, Y)

print(f"Input sentence: {source_sentence}")
print(f"Target we're explaining: {target_sentence}")
print(f"Calculated log-odds: {logodds}")

## Working with "Black-Box" models

Even if you don't have a Transformer object (maybe you're using a web API or a custom function), you can still use this wrapper. You just need to provide a "similarity model" that SHAP can use to calculate the scores.

In [3]:
# Imagine this is an external API call.
def my_custom_model(x):
    return ["This is a custom output!"]


# In this case, we use our local model as a 'similarity model' to help SHAP understand the scores.
tf_agnostic = shap.models.TeacherForcing(my_custom_model, similarity_model=model, similarity_tokenizer=tokenizer)

print("Agnostic setup ready to go!")